For this experimentation run we will freeze all layers except for the last three encoding layers, so in essence we will train four layers, the classification layer with the largest learning rate of 0.2, and the subsequently we will decrease the learning rate by a factor of 0.2. 

further areas of exploration:
- Use a lr decay such as a cosine schedular
- Increasing the trainable layers

In [20]:
import torchvision
import torch
import torch.nn as nn
import torchinfo
import wandb
import tqdm

In [29]:
lr = 0.2
lr_decay = 0.2
EPOCHS = 75 

In [18]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [48]:
def create_vit(device,
               lr, 
               lr_decay,
               num_classes: int = 100,
               seed: int = 42):
    weights = torchvision.models.ViT_B_32_Weights.DEFAULT
    model = torchvision.models.vit_b_32(weights=weights).to(device)

    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes)
    )

    layer_names = {
        "encoder_layer_9": [],
        "encoder_layer_10": [],
        "encoder_layer_11": [],
        "heads": []
    }

    for _, (name, param) in enumerate(model.named_parameters()):
        param.requires_grad = False
        for key in layer_names:
            if key in name:
                layer_names[key].append(name)
                param.requires_grad = True

    parameters = []

    for key in layer_names:
        parameters += [{'params': [p for p in layer_names[key]], 'lr': lr}]
        lr = lr * lr_decay

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

    return model, parameters

In [51]:
model, parameters = create_vit(device="mps", lr=lr, lr_decay=lr_decay)
model.to(device)

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [50]:
torchinfo.summary(model=model, 
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
VisionTransformer (VisionTransformer)                        [32, 3, 224, 224]    [32, 100]            768                  Partial
├─Conv2d (conv_proj)                                         [32, 3, 224, 224]    [32, 768, 7, 7]      (2,360,064)          False
├─Encoder (encoder)                                          [32, 50, 768]        [32, 50, 768]        38,400               Partial
│    └─Dropout (dropout)                                     [32, 50, 768]        [32, 50, 768]        --                   --
│    └─Sequential (layers)                                   [32, 50, 768]        [32, 50, 768]        --                   Partial
│    │    └─EncoderBlock (encoder_layer_0)                   [32, 50, 768]        [32, 50, 768]        (7,087,872)          False
│    │    └─EncoderBlock (encoder_layer_1)                   [32, 50, 768]        [

In [ ]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    wandb.init(project="AircraftDetection", config={"epochs": epochs, "model name":model_name})

    for epoch in tqdm(range(epochs)):
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)

        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            wandb.log({"test_loss": test_loss, "test_acc": test_acc, "train_loss": train_loss, "train_acc": train_acc})

    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        return results
    else:
        return results

In [ ]:
vit_loss_fn = torch.nn.CrossEntropyLoss()
vit_optimizer = torch.optim.Adam(params=parameters)

train(model=model,
      train_dataloader=vit_train_dataloder,
      test_dataloader=vit_test_dataloder,
      loss_fn=vit_loss_fn,
      learning_rate=lr,
      optimizer=vit_optimizer,
      device=device,
      epochs=EPOCHS,
      save_model=True,
      save_model_path="./models",
      model_name=f"vit_{EPOCHS}_epochs")